# 4. Efficiency, vetoes, backgrounds and constraints

**Learning goals:** describe selected data with consistent acceptance, fit a signal fraction, and add an external Gaussian constraint.

Run cells from top to bottom in a fresh Python kernel. Install the package and Jupyter
as explained in [the course guide](TUTORIALS.md). No external data files are needed.
Masses are in GeV, invariants in GeV², and daughter indices start at zero.
The small event counts and grid sizes keep this lesson practical on a CPU; they are
teaching settings, not a demonstrated precision choice for a physics analysis.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
from dalitzplotfitter import (
    DecayChannel, DecayModel, FitSession, NonResonant,
    Parameter, RealImag, Resonance, generate_toy,
)

In [2]:
channel = DecayChannel("D+", ("pi-", "pi+", "pi+"))
x = Parameter.coefficient("NR.x", 0.55, owner="NR", bounds=(-2, 2), step=0.02)
y = Parameter.coefficient("NR.y", 0.30, owner="NR", bounds=(-2, 2), step=0.02)
components = [
    Resonance("rho", (0, 1), RealImag(1, 0), mass=0.7753, width=0.1491, spin=1),
    NonResonant(RealImag(x, y), name="NR"),
]
model = DecayModel(
    channel, components, normalization_method="square-dalitz",
    normalization_resolution=100, normalization_pair=(0, 1),
)
truth = {p.name: p.value for p in model.parameters}

## Specify the observed distribution

The signal density is proportional to `efficiency * veto * abs(A)**2`. The same acceptance
must enter generation and fit normalization. Our positive smooth efficiency is illustrative.
The background below describes an already reconstructed shape; signal efficiency is not
automatically multiplied into it. Both categories share the mass-window veto.

The signal fraction is the fraction **after selection**. With one background its mixture
weight is `1 - signal_fraction`; `BackgroundSpec` computes its shape integral for us.

In [3]:
from dalitzplotfitter import (
    BackgroundSpec, ToyBackground, MassWindowVeto, GaussianConstraint,
)
def efficiency(events):
    return 0.45 + 0.40 * events["s23"] / channel.parent_mass**2

def background_shape(events):
    return jnp.ones_like(events["s12"])

veto = MassWindowVeto((0, 1), 0.45, 0.55)
fraction = Parameter("signal_fraction", 0.75, bounds=(0.05, 0.99), step=0.02)
data = generate_toy(
    model, 3500, parameters=truth, seed=41, efficiency=efficiency, veto=veto,
    signal_fraction=0.80, backgrounds=(ToyBackground("comb", background_shape),),
    inverse_resolution=384, include_momenta=False,
)
assert np.all(np.asarray(veto(data.as_dict())))
session = FitSession(
    model, data, efficiency=efficiency, veto=veto, signal_fraction=fraction,
    backgrounds=(BackgroundSpec("comb", background_shape),),
)
result = session.fit(simplex=True, ncall=5000)
session.report(result)
assert result.valid
session.plot_projection(result, "s12", bins=40, projection_size=25000)
plt.show()

## Add information from an independent measurement

This section uses the complete B+ -> pi+ pi- pi+ paper-style model. All numerical conventions, charge-dependent coefficients, and normalization choices follow the benchmark.

In [4]:
constrained = session.with_constraint(GaussianConstraint(fraction, mean=0.80, sigma=0.04))
constrained_result = constrained.fit(simplex=True, ncall=5000)
assert constrained_result.valid
for label, fitted in [("Unconstrained", result), ("Constrained", constrained_result)]:
    print(f"{label}: f_sig={fitted.values['signal_fraction']:.4f} "
          f"+/- {fitted.errors['signal_fraction']:.4f}")

For several backgrounds, the first N-1 fractions describe the relative background
composition and the last is the remainder. Extended fits instead use `signal_yield` and
per-background `yield_`. Histogram efficiency and background objects can replace the
callables here without changing this workflow. Their coordinate convention must match
the map production, and their values must not duplicate a grid Jacobian.

## Try it yourself

1. Increase the background fraction at generation and refit.
2. Compare acceptance-weighted and physical fit fractions using the report options.
3. Replace the constant background with a smooth positive function in both generation and fitting.

## Continue learning

[Next: dynamics and caching](tutorial_05_dynamics_and_low_level_api.ipynb). Reference: [backgrounds and vetoes](../../docs/backgrounds_and_vetoes.md), [constraints](../../docs/discriminants_and_constraints.md).

Return to [the course guide](TUTORIALS.md).